# EDA del Titanic con cuDF en Google Colab

Material en español para trabajar con **el CSV original del curso** (`titanic.csv`, 891 filas y 12 columnas). No se descarga ningún dataset desde Internet, no se limpia ni se imputan datos y no se eliminan filas. Esta práctica ilustra EDA acelerado en GPU; no sustituye cuDF por pandas ni afirma equivalencia experimental.

Documentación oficial: [Colab RAPIDS](https://docs.nvidia.com/datascience/deployment/stable/platforms/colab/), [instalación NVIDIA](https://docs.nvidia.com/datascience/install/) y [FAQ cuDF pandas](https://docs.rapids.ai/api/cudf/latest/cudf_pandas/faq/).

## 1. Comprobación y preparación del entorno

Antes de ejecutar nada, selecciona **Runtime > Change runtime type > GPU**. Ejecuta primero la celda de preflight siguiente. Si declara **LISTO PARA EJECUTAR CUDF**, no instales nada y continúa.

Si detecta una GPU NVIDIA pero cuDF no importa y declara **ERROR DE INSTALACIÓN**, ejecuta manualmente la celda opcional del instalador oficial de NVIDIA (enlace: [instalación NVIDIA](https://docs.nvidia.com/datascience/install/)). Si ya hubo imports, reinicia el runtime y vuelve a ejecutar desde arriba.

Si no hay GPU, no ejecutes el instalador ni intentes resolverlo localmente: conserva **NO EJECUCIÓN DE CUDF** y usa el fallback de referencia ya definido. Un error de instalación no se debe convertir en ‘sin GPU’.


In [ ]:
import sys, subprocess, shutil, traceback
print("Python:", sys.version)
resultado_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
print("nvidia-smi (retorno):", resultado_smi.returncode if resultado_smi else "no disponible")
print(resultado_smi.stdout if resultado_smi else "nvidia-smi no disponible")
if resultado_smi and resultado_smi.stderr:
    print("stderr nvidia-smi:", resultado_smi.stderr)

HARDWARE_NVIDIA_DETECTADO = resultado_smi is not None and resultado_smi.returncode == 0
GPU_CUDA_DISPONIBLE = False
CUDF_IMPORTADO = False
df = None
try:
    import cupy as cp
    dispositivos = int(cp.cuda.runtime.getDeviceCount())
    print("Dispositivos CUDA según CuPy:", dispositivos)
    if dispositivos > 0:
        prueba_gpu = cp.zeros(1)
        cp.cuda.runtime.deviceSynchronize()
        GPU_CUDA_DISPONIBLE = True
        nombre_gpu = cp.cuda.runtime.getDeviceProperties(0)['name']
        print("GPU/device:", nombre_gpu.decode() if isinstance(nombre_gpu, bytes) else nombre_gpu)
        del prueba_gpu
except Exception as e:
    print("Consulta CUDA con CuPy no disponible:", repr(e))

try:
    import cudf
    print("cuDF:", cudf.__version__)
    CUDF_IMPORTADO = True
except Exception as e:
    cudf = None
    CUDF_IMPORTADO = False
    ERROR_CUDF = repr(e)
    print("Error de importación cuDF (no ocultar):", repr(e))

if GPU_CUDA_DISPONIBLE and CUDF_IMPORTADO:
    ESTADO_CUDF = "LISTO PARA EJECUTAR CUDF"
elif HARDWARE_NVIDIA_DETECTADO or GPU_CUDA_DISPONIBLE:
    ESTADO_CUDF = "ERROR DE INSTALACIÓN"
else:
    ESTADO_CUDF = "GPU no disponible (NO EJECUCIÓN DE CUDF)"
    if CUDF_IMPORTADO:
        print("cuDF es importable, pero no hay dispositivo CUDA: no se ejecuta cuDF.")
print("Estado declarado:", ESTADO_CUDF)

In [ ]:
# Celda opcional/manual: ejecútala solo con GPU NVIDIA y ERROR DE INSTALACIÓN.
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py


### Si cuDF no está instalado

No ejecutes un instalador alternativo de esta práctica. En una sesión Colab compatible, sigue el instalador oficial de NVIDIA/RAPIDS mediante el script `rapidsai-csp-utils`; después **reinicia el runtime si hubo imports previos** y vuelve a ejecutar desde el principio. Si falla, conserva y documenta el error: eso es el estado **ERROR DE INSTALACIÓN**, no una ejecución de cuDF.

## 2. Cargar el CSV original del curso

Usa el archivo proporcionado por el curso, no una descarga ni una copia transformada. La primera opción permite subirlo desde tu equipo. La segunda monta Drive para seleccionar el mismo `titanic.csv` que hayas guardado allí. Ajusta solo `RUTA_CSV`; no cambies el contenido del CSV.

In [ ]:
RUTA_CSV = None  # None permite subir titanic.csv; para Drive, asignad su ruta.

# Opción B (en vez de la anterior): Drive
# from google.colab import drive
# drive.mount('/content/drive')
# RUTA_CSV = '/content/drive/MyDrive/ruta/al/dataset-del-curso/titanic.csv'

if ESTADO_CUDF == "LISTO PARA EJECUTAR CUDF":
    try:
        if RUTA_CSV is None:
            from google.colab import files
            subidos = files.upload()
            if 'titanic.csv' not in subidos:
                raise ValueError('Sube el CSV original del curso con el nombre titanic.csv.')
            RUTA_CSV = 'titanic.csv'
        from pathlib import Path
        from hashlib import sha256
        df = cudf.read_csv(RUTA_CSV)
        HUELLA_CSV = sha256(Path(RUTA_CSV).read_bytes()).hexdigest()
        ESTADO_CUDF = "GPU/cuDF ejecutado"
        print("Archivo:", RUTA_CSV)
        print("SHA-256:", HUELLA_CSV)
        print("Shape:", df.shape)
        print(df.dtypes)
    except Exception as e:
        ESTADO_CUDF = "ERROR DE CARGA O EJECUCIÓN CUDF"
        ERROR_CARGA_CUDF = repr(e)
        df = None
        traceback.print_exc()
        print(ESTADO_CUDF, ERROR_CARGA_CUDF)
else:
    print("NO EJECUCIÓN DE CUDF: no se solicita archivo ni se llama a read_csv. Estado:", ESTADO_CUDF)

## 3. Estado sin GPU o con instalación fallida

La tabla siguiente son **valores de referencia precomputados del CSV original**, no una ejecución alternativa. Si el estado anterior no fue `GPU/cuDF ejecutado`, no ejecutes ni presentes como resultados los bloques cuDF: conserva literalmente la etiqueta **NO EJECUCIÓN DE CUDF** o **ERROR DE INSTALACIÓN** y vuelve a intentar en un runtime compatible.

In [ ]:
CONTADORES_REFERENCIA = {
    'global': {'todos': (891, 342)},
    'Sex': {'female': (314, 233), 'male': (577, 109)},
    'Pclass': {1: (216, 136), 2: (184, 87), 3: (491, 119)},
    'Pclass x Sex': {(1, 'female'): (94, 91), (1, 'male'): (122, 45), (2, 'female'): (76, 70), (2, 'male'): (108, 17), (3, 'female'): (144, 72), (3, 'male'): (347, 47)},
    'grupo_edad': {'<18': (113, 61), '>=18': (601, 229)},
}
REFERENCIA = [dict(analisis=analisis, grupo=grupo, total=total, supervivientes=supervivientes, tasa=supervivientes/total)
              for analisis, grupos in CONTADORES_REFERENCIA.items() for grupo, (total, supervivientes) in grupos.items()]
if ESTADO_CUDF != "GPU/cuDF ejecutado":
    print(ESTADO_CUDF)
    print("NO EJECUCIÓN DE CUDF — REFERENCIA PRECOMPUTADA, no resultados propios:")
    for fila in REFERENCIA:
        print(fila)

## 4. Estructura y nulos (sin limpieza)

In [ ]:
if ESTADO_CUDF == "GPU/cuDF ejecutado":
    print("shape:", df.shape)
    columnas_esperadas = ["PassengerId", "Survived", "Pclass", "Name", "Sex", "Age", "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked"]
    assert list(df.columns) == columnas_esperadas
    print("schema:")
    print(df.dtypes)
    print("nulos por columna:")
    print(df.isnull().sum())
    assert tuple(df.shape) == (891, 12)
    assert int(df["Age"].isnull().sum()) == 177
    assert int(df["Cabin"].isnull().sum()) == 687
    assert int(df["Embarked"].isnull().sum()) == 2
else:
    print("NO EJECUCIÓN DE CUDF: estructura y nulos no calculados en esta sesión.")

## 5. Tablas de supervivencia

Cada tabla muestra las claves y `total` (pasajeros del grupo), `supervivientes` y `tasa = supervivientes/total`, con el mismo significado que Polars y DuckDB. Se conserva el dataset crudo. Solo el análisis por edad excluye Age nulo, porque no puede clasificarse en ningún intervalo.

In [ ]:
def resumen_supervivencia(tabla, por):
    g = tabla.groupby(por)['Survived'].agg(['size', 'sum']).reset_index()
    g = g.rename(columns={'size': 'total', 'sum': 'supervivientes'})
    g['tasa'] = g['supervivientes'] / g['total']
    return g[por + ['total', 'supervivientes', 'tasa']].sort_values(por).reset_index(drop=True)

if ESTADO_CUDF == "GPU/cuDF ejecutado":
    global_eda = cudf.DataFrame({'total': [len(df)], 'supervivientes': [int(df['Survived'].sum())]})
    global_eda['tasa'] = global_eda['supervivientes'] / global_eda['total']
    por_sexo = resumen_supervivencia(df, ['Sex'])
    por_clase = resumen_supervivencia(df, ['Pclass'])
    por_clase_sexo = resumen_supervivencia(df, ['Pclass', 'Sex'])
    print('Global:\n', global_eda)
    print('Sexo (female, male):\n', por_sexo)
    print('Clase (1, 2, 3):\n', por_clase)
    print('Clase y sexo:\n', por_clase_sexo)
    edad_df = df[df["Age"].notnull()].copy()
    menores = edad_df["Age"].lt(18)
    grupo_edad = cudf.Series([">=18"] * len(edad_df), index=edad_df.index)
    grupo_edad[menores] = "<18"
    edad_df["grupo_edad"] = grupo_edad
    por_edad = resumen_supervivencia(edad_df, ['grupo_edad'])
    print('Edad (<18, >=18):\n', por_edad)
else:
    print("NO EJECUCIÓN DE CUDF: tablas no calculadas. Usa solo la tabla de referencia anterior.")

## 6. Assertions de control

Estas comprobaciones son exactas para el CSV original. La tolerancia de tasa es `1e-6`; los conteos deben coincidir exactamente.

In [ ]:
if ESTADO_CUDF == "GPU/cuDF ejecutado":
    def comprueba(tabla_, columnas, esperados):
        assert list(tabla_.columns) == columnas + ['total', 'supervivientes', 'tasa']
        assert len(tabla_) == len(esperados)
        indexada = tabla_.set_index(columnas)
        for clave, (total, supervivientes) in esperados.items():
            fila = indexada.loc[clave]
            assert int(fila['total']) == total and int(fila['supervivientes']) == supervivientes
            assert abs(float(fila['tasa']) - supervivientes / total) <= 1e-6
    assert int(df["Survived"].sum()) == 342 and len(df) == 891
    assert list(global_eda.columns) == ['total', 'supervivientes', 'tasa']
    assert int(global_eda['total'].iloc[0]) == 891 and int(global_eda['supervivientes'].iloc[0]) == 342
    comprueba(por_sexo, ['Sex'], CONTADORES_REFERENCIA['Sex'])
    comprueba(por_clase, ['Pclass'], CONTADORES_REFERENCIA['Pclass'])
    comprueba(por_clase_sexo, ['Pclass', 'Sex'], CONTADORES_REFERENCIA['Pclass x Sex'])
    comprueba(por_edad, ['grupo_edad'], CONTADORES_REFERENCIA['grupo_edad'])
    assert int(por_edad['total'].sum()) == 714
    print("Assertions superadas: conteos globales y tasas con tolerancia 1e-6.")
else:
    print("NO EJECUCIÓN DE CUDF: assertions no ejecutadas.")

## 7. Interpretación responsable

Estas tablas describen **asociaciones** observadas en este conjunto histórico; no demuestran que el sexo, la clase o la edad causaran la supervivencia. Hay posibles variables mezcladas y sesgos de selección. No hacemos gráficos, modelos de ML ni benchmarks en esta práctica.

## 8. Pregunta propia del equipo experto

Formulad una pregunta adicional y preparad su código cuDF con filtro/agrupación y columnas total/supervivientes/tasa. Explicad población, denominador y tratamiento de ausentes; no alteréis ni imputéis el original.

**Pregunta:** por completar.
**Filtro, población y nulos:** por completar.
**Interpretación y limitación:** por completar.

Si no hay GPU o hay un error, el código y el resultado se marcan PENDIENTE DE EJECUCIÓN. Las referencias precomputadas no son vuestra pregunta propia ejecutada. Entrega: [entrega-comparativa-motores-titanic.md](entrega-comparativa-motores-titanic.md).

In [ ]:
if ESTADO_CUDF == 'GPU/cuDF ejecutado':
    # TODO: construir tabla derivada y reutilizar resumen_supervivencia(...).
    pass
else:
    print('PREGUNTA PROPIA PENDIENTE DE EJECUCIÓN. Estado:', ESTADO_CUDF)
